In [ ]:
#============================================================
# Experiment 9A: K-Means Clustering from Scratch
# Synthetic Customer Dataset (Income vs Spending Score)
#============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Create Dataset
# X1 = Annual Income (Lakhs)
# X2 = Spending Score
X = np.array([
    [2.0, 2.5],
    [2.5, 3.0],
    [3.0, 2.0],
    [3.5, 3.5],

    [7.0, 8.0],
    [7.5, 7.0],
    [8.0, 8.5],
    [8.5, 7.5],

    [5.0, 2.0],
    [5.5, 2.5],
    [6.0, 1.5],
    [6.5, 2.0]
])

# Step 2: Display Dataset
print("Dataset")
print("================================")
print("Income\tSpending Score")
for row in X:
    print(f"{row[0]:.1f}\t{row[1]:.1f}")

# Step 3: Set Number of Clusters
K = 3

# Step 4: Initialize Centroids (select three initial observations)
centroids = np.array([X[0], X[4], X[8]])
print("\nInitial Centroids:")
print(centroids)

# Step 5: Calculate Euclidean Distance
def euclidean_distance(point, centroid):
    return np.sqrt(np.sum((point - centroid)**2))

# Step 6: Assign Data Points to Clusters
def assign_clusters(X, centroids):
    clusters = []
    for point in X:
        distances = []
        for centroid in centroids:
            distance = euclidean_distance(point, centroid)
            distances.append(distance)
        cluster = np.argmin(distances)
        clusters.append(cluster)
    return np.array(clusters)

# Step 7: Update Centroids
def update_centroids(X, clusters, K):
    new_centroids = []
    for k in range(K):
        cluster_points = X[clusters == k]
        if len(cluster_points) > 0:
            centroid = np.mean(cluster_points, axis=0)
        else:
            centroid = X[np.random.randint(len(X))]
        new_centroids.append(centroid)
    return np.array(new_centroids)

# Step 8: K-Means Algorithm
max_iterations = 100
tolerance = 0.0001

for iteration in range(max_iterations):

    # Assign points to nearest centroid
    clusters = assign_clusters(X, centroids)

    # Calculate new centroids
    new_centroids = update_centroids(X, clusters, K)

    # Calculate centroid movement
    centroid_shift = np.linalg.norm(new_centroids - centroids)

    print(f"\nIteration {iteration + 1}")
    print("Centroids:")
    print(new_centroids)
    print("Centroid Shift:", round(centroid_shift, 6))

    # Check convergence
    centroids = new_centroids
    if centroid_shift < tolerance:
        print("\nK-Means Converged.")
        break

# Step 9: Final Cluster Assignment
clusters = assign_clusters(X, centroids)

# Step 10: Display Final Results
print("\n===================================")
print("FINAL K-MEANS RESULTS")
print("===================================")
for i in range(K):
    print(f"\nCluster {i + 1}")
    print(X[clusters == i])

print("\nFinal Centroids:")
print(centroids)

# Step 11: Calculate Within-Cluster Sum of Squares (WCSS)
wcss = 0
for k in range(K):
    cluster_points = X[clusters == k]
    distances = cluster_points - centroids[k]
    wcss += np.sum(distances**2)

print("\nWithin-Cluster Sum of Squares:")
print(round(wcss, 4))

# Step 12: Predict Cluster for New Customer
new_customer = np.array([[6.2, 2.2]])
new_cluster = assign_clusters(new_customer, centroids)[0]

print("\n===================================")
print("NEW CUSTOMER PREDICTION")
print("===================================")
print("Annual Income:", new_customer[0, 0], "Lakhs")
print("Spending Score:", new_customer[0, 1])
print("Predicted Cluster:", new_cluster + 1)

# Step 13: Plot Clusters
plt.figure(figsize=(8, 6))

for k in range(K):
    plt.scatter(X[clusters == k, 0], X[clusters == k, 1],
                label=f"Cluster {k + 1}")

# Plot centroids
plt.scatter(centroids[:, 0], centroids[:, 1],
            marker="X", s=200, label="Centroids")

# Plot new customer
plt.scatter(new_customer[0, 0], new_customer[0, 1],
            marker="*", s=200, label="New Customer")

plt.xlabel("Annual Income (Lakhs)")
plt.ylabel("Spending Score")
plt.title("K-Means Clustering From Scratch")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Experiment 9B: DBSCAN Clustering using Scikit-learn
# Real-World Dataset: Iris Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Step 1: Load Real-World Dataset
iris = load_iris()
X = iris.data
feature_names = iris.feature_names
target = iris.target
target_names = iris.target_names

print("Dataset Shape:")
print(X.shape)

# Step 2: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Actual Class"] = target
df["Class Name"] = [target_names[i] for i in target]

print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nFeature Names:")
print(feature_names)
print("\nDataset Description:")
print(df.describe())

# Step 4: Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 5: Create DBSCAN Model
model = DBSCAN(
    eps=0.8,
    min_samples=5,
    metric="euclidean"
)

# Step 6: Apply DBSCAN
cluster_labels = model.fit_predict(X_scaled)

# Step 7: Add Cluster Labels
df["DBSCAN Cluster"] = cluster_labels
print("\nClustered Dataset:")
print(df.head(10))

# Step 8: Determine Number of Clusters
unique_labels = set(cluster_labels)
n_clusters = len(unique_labels - {-1})
n_noise = np.sum(cluster_labels == -1)

print("\n===================================")
print("DBSCAN RESULTS")
print("===================================")
print("Number of Clusters:", n_clusters)
print("Number of Noise Points:", n_noise)

# Step 9: Cluster Distribution
cluster_counts = (
    df["DBSCAN Cluster"]
    .value_counts()
    .sort_index()
)
print("\nCluster Distribution:")
print(cluster_counts)

# Step 10: Silhouette Score (non-noise observations only)
non_noise = (cluster_labels != -1)

if n_clusters >= 2:
    silhouette = silhouette_score(
        X_scaled[non_noise],
        cluster_labels[non_noise]
    )
    print("\nSilhouette Score:", round(silhouette, 4))
else:
    print("\nSilhouette Score:")
    print("Cannot be calculated.")

# Step 11: Visualize DBSCAN Clusters
plt.figure(figsize=(9, 6))

for cluster in sorted(unique_labels):
    cluster_points = (cluster_labels == cluster)
    if cluster == -1:
        label = "Noise"
    else:
        label = f"Cluster {cluster}"
    plt.scatter(X[cluster_points, 0], X[cluster_points, 1], label=label)

plt.xlabel("Sepal Length (cm)")
plt.ylabel("Sepal Width (cm)")
plt.title("DBSCAN Clustering - Iris Dataset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 12: Display Actual vs DBSCAN Cluster
comparison = pd.DataFrame({
    "Actual Class": target,
    "DBSCAN Cluster": cluster_labels
})
print("\nActual Class vs DBSCAN Cluster:")
print(comparison.head(20))

# Step 13: Save Results
df.to_csv("Iris_DBSCAN_Results.csv", index=False)
print("\nResults saved as:")
print("Iris_DBSCAN_Results.csv")

In [ ]:
#============================================================
# Experiment 9C: Hierarchical Agglomerative Clustering
# Real-World Dataset: Iris Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Step 1: Load Real-World Dataset
iris = load_iris()
X = iris.data
feature_names = iris.feature_names
target = iris.target
target_names = iris.target_names

print("Dataset Shape:")
print(X.shape)

# Step 2: Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Actual Class"] = target
df["Class Name"] = [target_names[i] for i in target]

print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nFeature Names:")
print(feature_names)
print("\nDataset Description:")
print(df.describe())

# Step 4: Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 5: Create Dendrogram (Ward Linkage)
linkage_matrix = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=90,
    leaf_font_size=10
)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Data Points / Clusters")
plt.ylabel("Distance")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 6: Create Agglomerative Clustering Model
n_clusters = 3
model = AgglomerativeClustering(
    n_clusters=n_clusters,
    metric="euclidean",
    linkage="ward"
)

# Step 7: Apply Hierarchical Clustering
cluster_labels = model.fit_predict(X_scaled)

# Step 8: Add Cluster Labels
df["Cluster"] = cluster_labels + 1
print("\nClustered Dataset:")
print(df.head(10))

# Step 9: Display Cluster Distribution
cluster_counts = (
    df["Cluster"]
    .value_counts()
    .sort_index()
)

print("\n===================================")
print("HIERARCHICAL CLUSTERING RESULTS")
print("===================================")
print("Number of Clusters:", n_clusters)
print("\nCluster Distribution:")
print(cluster_counts)

# Step 10: Calculate Silhouette Score
silhouette = silhouette_score(X_scaled, cluster_labels)
print("\nSilhouette Score:", round(silhouette, 4))

# Step 11: Cluster Summary (mean feature values per cluster)
cluster_summary = df.groupby("Cluster")[feature_names].mean()
print("\nCluster Summary:")
print(cluster_summary)

# Step 12: Visualize Clusters
plt.figure(figsize=(9, 6))

for cluster in range(1, n_clusters + 1):
    cluster_points = (df["Cluster"] == cluster)
    plt.scatter(X[cluster_points, 0], X[cluster_points, 1],
                label=f"Cluster {cluster}")

plt.xlabel("Sepal Length (cm)")
plt.ylabel("Sepal Width (cm)")
plt.title("Hierarchical Clustering - Iris Dataset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 13: Actual Class vs Cluster
comparison = pd.DataFrame({
    "Actual Class": [target_names[i] for i in target],
    "Cluster": cluster_labels + 1
})
print("\nActual Class vs Cluster:")
print(comparison.head(20))

# Step 14: Crosstab
crosstab = pd.crosstab(comparison["Actual Class"], comparison["Cluster"])
print("\nActual Class vs Cluster Table:")
print(crosstab)

# Step 15: Save Results
df.to_csv("Iris_Hierarchical_Clustering_Results.csv", index=False)
print("\nResults saved as:")
print("Iris_Hierarchical_Clustering_Results.csv")